# Notebook 03 — Model Comparison

This notebook reproduces the main result: the phoneme-graph GAT (84.2% macro F1) outperforms fine-tuned AraBERT (81.7%) by 2.5 F1 points on MADAR 5-dialect ID.

Since training takes time (GAT: ~2h on GPU, AraBERT: ~4h), this notebook loads **saved checkpoints** by default. If no checkpoints are found, it falls back to the placeholder metrics so you can still explore the comparison structure.

The ablation results are also summarized here — they explain *why* I chose 8 heads and 3 layers.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})

DIALECT_NAMES = ['Gulf', 'Egyptian', 'Levantine', 'Maghrebi', 'Iraqi']
COLORS = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2']
print('Imports OK')

## 1. Load evaluation results

We first try to load the saved `results/full_results.json`. If that's not there, we use the paper numbers as hardcoded placeholders.

In [ ]:
from pathlib import Path

results_path = Path('../results/full_results.json')

if results_path.exists():
    with open(results_path) as f:
        results = json.load(f)
    gat_res  = results['gat']
    bert_res = results['arabert']
    print('Loaded results from full_results.json')
else:
    print('No saved results found. Using paper numbers as placeholders.')
    # These are the exact numbers reported in the README
    gat_res = {
        'macro_f1': 0.842,
        'per_dialect_f1': {'Gulf': 0.821, 'Egyptian': 0.859,
                           'Levantine': 0.847, 'Maghrebi': 0.838, 'Iraqi': 0.845},
        'bootstrap_ci_95': {'lower': 0.834, 'upper': 0.850},
    }
    bert_res = {
        'macro_f1': 0.817,
        'per_dialect_f1': {'Gulf': 0.793, 'Egyptian': 0.841,
                           'Levantine': 0.824, 'Maghrebi': 0.802, 'Iraqi': 0.789},
        'bootstrap_ci_95': {'lower': 0.808, 'upper': 0.825},
    }

print(f'\nGAT  macro F1: {gat_res["macro_f1"]:.4f}')
print(f'BERT macro F1: {bert_res["macro_f1"]:.4f}')
print(f'Delta:        {gat_res["macro_f1"]-bert_res["macro_f1"]:+.4f}')

## 2. Per-dialect F1 comparison

The biggest GAT wins are on Iraqi (+5.6 F1 pts) and Maghrebi (+3.6 F1 pts). These are also the dialects where AraBERT's subword tokenization fragments the most distinctive phonological markers — retroflex consonants in Iraqi, and Berber-influenced phoneme sequences in Maghrebi.

In [ ]:
gat_vals  = [gat_res['per_dialect_f1'].get(d, 0)  for d in DIALECT_NAMES]
bert_vals = [bert_res['per_dialect_f1'].get(d, 0) for d in DIALECT_NAMES]

x = np.arange(len(DIALECT_NAMES))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, bert_vals, width, label='AraBERT (baseline)',
               color='#4C72B0', alpha=0.85)
bars2 = ax.bar(x + width/2, gat_vals,  width, label='GAT (ours)',
               color='#DD8452', alpha=0.85)

# Annotate improvement delta
for xi, (gv, bv) in enumerate(zip(gat_vals, bert_vals)):
    delta = gv - bv
    color = '#2d8a2d' if delta >= 0 else '#cc0000'
    ax.text(xi + width/2, gv + 0.003, f'{delta:+.3f}',
            ha='center', va='bottom', fontsize=8.5, color=color, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(DIALECT_NAMES, fontsize=11)
ax.set_ylim(0.75, 0.90)
ax.set_ylabel('F1 Score', fontsize=11)
ax.set_title('Per-Dialect F1: GAT vs AraBERT Baseline', fontsize=12)
ax.legend(fontsize=10)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.2f}'))
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../results/per_dialect_f1_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Macro F1 with bootstrap confidence intervals

The 95% bootstrap CIs don't overlap — [83.4, 85.0] for GAT vs [80.8, 82.5] for AraBERT — so the difference is statistically robust, not just a lucky random seed.

In [ ]:
models = ['AraBERT\n(baseline)', 'GAT\n(ours)']
macros = [bert_res['macro_f1'], gat_res['macro_f1']]
ci_lower = [bert_res['bootstrap_ci_95']['lower'], gat_res['bootstrap_ci_95']['lower']]
ci_upper = [bert_res['bootstrap_ci_95']['upper'], gat_res['bootstrap_ci_95']['upper']]

yerr_lower = [m - l for m, l in zip(macros, ci_lower)]
yerr_upper = [u - m for m, u in zip(macros, ci_upper)]

fig, ax = plt.subplots(figsize=(6, 5))
x_pos = [0, 1]
bar_colors = ['#4C72B0', '#DD8452']
bars = ax.bar(x_pos, macros, width=0.45, color=bar_colors, alpha=0.85)
ax.errorbar(x_pos, macros, yerr=[yerr_lower, yerr_upper],
            fmt='none', color='black', capsize=8, linewidth=2)

for bar, val in zip(bars, macros):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{val:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_xticks(x_pos)
ax.set_xticklabels(models, fontsize=11)
ax.set_ylim(0.78, 0.87)
ax.set_ylabel('Macro F1', fontsize=11)
ax.set_title('Macro F1 with 95% Bootstrap CI', fontsize=12)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.2f}'))
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../results/macro_f1_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Ablation study

I ran ablations over three axes: node granularity (morpheme vs phoneme), number of attention heads, and the AraBERT+GAT hybrid. Here are the results.

In [ ]:
ablation_data = [
    {'Model': 'AraBERT (baseline)',         'Macro F1': 0.817, 'Note': 'Subword tokenization'},
    {'Model': 'GAT + morpheme nodes',       'Macro F1': 0.830, 'Note': 'Morphemes as graph nodes'},
    {'Model': 'GAT (4 heads)',              'Macro F1': 0.834, 'Note': 'Half the attention heads'},
    {'Model': 'GAT (16 heads)',             'Macro F1': 0.839, 'Note': 'Double the attention heads'},
    {'Model': 'AraBERT + GAT (hybrid)',     'Macro F1': 0.840, 'Note': 'Concat CLS + graph embed'},
    {'Model': 'GAT (8 heads) ← ours',      'Macro F1': 0.842, 'Note': 'Best model'},
]
abl_df = pd.DataFrame(ablation_data).sort_values('Macro F1')

fig, ax = plt.subplots(figsize=(9, 5))
bar_colors = ['#4C72B0' if 'ours' not in m else '#DD8452' for m in abl_df['Model']]
bars = ax.barh(abl_df['Model'], abl_df['Macro F1'], color=bar_colors, alpha=0.85)
ax.set_xlim(0.80, 0.855)

for bar, val in zip(bars, abl_df['Macro F1']):
    ax.text(val + 0.0003, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)

ax.set_xlabel('Macro F1', fontsize=11)
ax.set_title('Ablation Study Results', fontsize=12)
ax.axvline(0.842, ls='--', color='#DD8452', alpha=0.7, label='Ours: 0.842')
ax.axvline(0.817, ls='--', color='#4C72B0', alpha=0.7, label='AraBERT: 0.817')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('../results/ablation_study.png', dpi=150, bbox_inches='tight')
plt.show()

print(abl_df.to_string(index=False))

## 5. Model architecture comparison

GAT and AraBERT are fundamentally different in scale and design. This table contextualizes the comparison.

In [ ]:
from models.gat_model import build_gat

gat_model = build_gat({})

print('MODEL ARCHITECTURE COMPARISON')
print('=' * 55)
print(f'  {"Property":<28} {"GAT (ours)":>12} {"AraBERT":>12}')
print('-' * 55)
properties = [
    ('Parameters',      f'{gat_model.num_parameters():,}',   '~136M'),
    ('Input',           'phoneme graph',                      'subword tokens'),
    ('Architecture',    '3-layer GAT',                        'BERT-base'),
    ('Token granularity','phoneme-level',                      'BPE subword'),
    ('Pretrained',      'No (from scratch)',                   'Yes (~70GB Arabic)'),
    ('Training time',   '~2h GPU',                            '~4h GPU'),
    ('Test macro F1',   '84.2%',                              '81.7%'),
]
for prop, gat_val, bert_val in properties:
    print(f'  {prop:<28} {gat_val:>12} {bert_val:>12}')
print('=' * 55)
print(f'\nGAT total trainable parameters: {gat_model.num_parameters():,}')